# Daily Challenge: Classifying Handwritten Digits with CNNs
## Week 6 — Day 2 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build and compare two models for MNIST digit classification:
- A **Fully Connected Neural Network** (Dense layers only)
- A **Convolutional Neural Network** (CNN)

**Steps:**
1. Load & explore MNIST
2. Preprocess for Fully Connected NN → Build & Train
3. Preprocess for CNN → Build & Train
4. Compare performance

> **Run on Google Colab** — TensorFlow and GPU acceleration required.

## Step 1 — Setup & Load MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow {tf.__version__} ✓")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

tf.random.set_seed(42)
np.random.seed(42)

# Load MNIST
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print(f"\nDataset loaded ✓")
print(f"X_train shape : {X_train_raw.shape}  — {X_train_raw.shape[0]} images of {X_train_raw.shape[1]}×{X_train_raw.shape[2]} pixels")
print(f"y_train shape : {y_train_raw.shape}  — integer labels 0-9")
print(f"X_test  shape : {X_test_raw.shape}")
print(f"y_test  shape : {y_test_raw.shape}")
print(f"Pixel value range: [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Classes: {np.unique(y_train_raw)}")

# Visualize sample digits
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]
    axes[0, digit].imshow(X_train_raw[idx], cmap='gray')
    axes[0, digit].set_title(str(digit), fontsize=11, fontweight='bold')
    axes[0, digit].axis('off')
    idx2 = np.where(y_train_raw == digit)[0][1]
    axes[1, digit].imshow(X_train_raw[idx2], cmap='gray')
    axes[1, digit].axis('off')

plt.suptitle('MNIST Sample Images (2 per digit)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot1_mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 2 — Preprocess for Fully Connected Neural Network

In [ ]:
# Flatten: 28×28 → 784
X_train_flat = X_train_raw.reshape(-1, 28 * 28).astype('float32') / 255.0
X_test_flat  = X_test_raw.reshape(-1, 28 * 28).astype('float32') / 255.0

# One-hot encode labels
y_train_oh = to_categorical(y_train_raw, num_classes=10)
y_test_oh  = to_categorical(y_test_raw,  num_classes=10)

print("Preprocessing for Fully Connected NN ✓")
print(f"  X_train_flat shape : {X_train_flat.shape}  (each image → 784-d vector)")
print(f"  X_test_flat  shape : {X_test_flat.shape}")
print(f"  y_train_oh   shape : {y_train_oh.shape}  (one-hot: 10 classes)")
print(f"  y_test_oh    shape : {y_test_oh.shape}")
print(f"  Pixel range: [{X_train_flat.min():.2f}, {X_train_flat.max():.2f}]  (normalized)")
print(f"\nExample label — digit '{y_train_raw[0]}' → one-hot: {y_train_oh[0].astype(int)}")

## Step 3 — Build & Train Fully Connected Neural Network

In [ ]:
model_fc = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='fully_connected_nn')

model_fc.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_fc.summary()

history_fc = model_fc.fit(
    X_train_flat, y_train_oh,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

fc_train_loss, fc_train_acc = model_fc.evaluate(X_train_flat, y_train_oh, verbose=0)
fc_test_loss,  fc_test_acc  = model_fc.evaluate(X_test_flat,  y_test_oh,  verbose=0)

print(f"\nFully Connected NN Results:")
print(f"  Train Accuracy : {fc_train_acc*100:.2f}%")
print(f"  Test  Accuracy : {fc_test_acc*100:.2f}%")

## Step 4 — Preprocess for Convolutional Neural Network

In [ ]:
# Reshape: (N, 28, 28) → (N, 28, 28, 1)  — Conv2D expects channel dimension
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Labels stay one-hot (reuse y_train_oh, y_test_oh from above)
print("Preprocessing for CNN ✓")
print(f"  X_train_cnn shape : {X_train_cnn.shape}  (N, height, width, channels)")
print(f"  X_test_cnn  shape : {X_test_cnn.shape}")
print(f"  Pixel range: [{X_train_cnn.min():.2f}, {X_train_cnn.max():.2f}]  (normalized)")
print(f"\nKey difference from Fully Connected preprocessing:")
print("  FC  → reshape to (N, 784)       — loses spatial structure")
print("  CNN → reshape to (N, 28, 28, 1) — preserves 2D spatial structure")

## Step 5 — Build & Train Convolutional Neural Network

In [ ]:
model_cnn = keras.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(2, 2),
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='cnn_model')

model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

history_cnn = model_cnn.fit(
    X_train_cnn, y_train_oh,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

cnn_train_loss, cnn_train_acc = model_cnn.evaluate(X_train_cnn, y_train_oh, verbose=0)
cnn_test_loss,  cnn_test_acc  = model_cnn.evaluate(X_test_cnn,  y_test_oh,  verbose=0)

print(f"\nCNN Results:")
print(f"  Train Accuracy : {cnn_train_acc*100:.2f}%")
print(f"  Test  Accuracy : {cnn_test_acc*100:.2f}%")

## Step 6 — Compare Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].plot(history_fc.history['accuracy'],     label='FC Train',  color='#4e91d4', lw=2)
axes[0].plot(history_fc.history['val_accuracy'], label='FC Val',    color='#4e91d4', lw=2, linestyle='--')
axes[0].plot(history_cnn.history['accuracy'],    label='CNN Train', color='#e05c5c', lw=2)
axes[0].plot(history_cnn.history['val_accuracy'],label='CNN Val',   color='#e05c5c', lw=2, linestyle='--')
axes[0].set_title('Accuracy — FC vs CNN', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss comparison
axes[1].plot(history_fc.history['loss'],     label='FC Train',  color='#4e91d4', lw=2)
axes[1].plot(history_fc.history['val_loss'], label='FC Val',    color='#4e91d4', lw=2, linestyle='--')
axes[1].plot(history_cnn.history['loss'],    label='CNN Train', color='#e05c5c', lw=2)
axes[1].plot(history_cnn.history['val_loss'],label='CNN Val',   color='#e05c5c', lw=2, linestyle='--')
axes[1].set_title('Loss — FC vs CNN', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Fully Connected vs CNN — Training Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot2_fc_vs_cnn_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
models_names = ['Fully Connected NN', 'CNN']
train_accs   = [fc_train_acc * 100, cnn_train_acc * 100]
test_accs    = [fc_test_acc  * 100, cnn_test_acc  * 100]

x = np.arange(len(models_names))
width = 0.35
bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='#4e91d4')
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',  color='#e05c5c')

for bar, val in zip(bars1, train_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(models_names, fontsize=11)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(95, 100.5)
ax.set_title('Model Accuracy Comparison', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('dc_plot3_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# Summary table
print(f"\n{'='*55}")
print(f"  {'MODEL':<25} {'TRAIN ACC':>10} {'TEST ACC':>10}")
print(f"{'='*55}")
print(f"  {'Fully Connected NN':<25} {fc_train_acc*100:>9.2f}% {fc_test_acc*100:>9.2f}%")
print(f"  {'CNN':<25} {cnn_train_acc*100:>9.2f}% {cnn_test_acc*100:>9.2f}%")
print(f"{'='*55}")
print(f"  CNN improvement  : {(cnn_test_acc - fc_test_acc)*100:+.2f}% on test set")
print(f"{'='*55}")

## Summary & Key Takeaways

**What I learned:**

1. **Data preprocessing differs between FC and CNN:**
   - FC NN requires flattening images from 28×28 → 784 (loses spatial structure)
   - CNN keeps the 2D shape (28×28×1) — it explicitly exploits pixel neighbourhood relationships

2. **Fully Connected NN vs CNN on MNIST:**
   - FC NN (~98% test accuracy): treats each pixel independently, misses local patterns like curves and edges
   - CNN (~99%+ test accuracy): Conv2D + MaxPooling layers detect local features (edges, strokes) regardless of position in the image

3. **Why CNN outperforms FC on images:**
   - **Parameter sharing**: one filter scans the entire image → fewer parameters, better generalization
   - **Translation invariance**: MaxPooling makes the model robust to slight shifts/distortions in digit position
   - **Hierarchical features**: early layers → edges/curves; later layers → digit shapes

4. **Architecture decisions:**
   - Two Conv2D+MaxPool blocks are sufficient for 28×28 grayscale images
   - Flatten then Dense layers form the classification head after feature extraction
   - Softmax with 10 outputs + categorical crossentropy loss = standard multi-class setup

5. **Same number of epochs, better accuracy**: CNNs are not just more accurate — they also converge faster and generalize better, showing lower validation loss throughout training.